In [3]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [4]:
!pip install opencv-python mediapipe scikit-learn

In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [6]:
def printimg(img,rev=True):
    if rev:
        img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [7]:
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.HandLandmarker.create_from_options(options)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1782810846.577401     312 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782810846.602040     312 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [51]:
DATA_DIR = "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
x_file_path = '/kaggle/working/X_data.npy'
y_file_path = '/kaggle/working/y_labels.npy'

In [59]:
import os

# Define the file paths
x_file_path = '/kaggle/working/X_data.npy'
y_file_path = '/kaggle/working/y_labels.npy'

# Delete X_data.npy if it exists
if os.path.exists(x_file_path):
    os.remove(x_file_path)
    print(f"Deleted: {x_file_path}")
else:
    print(f"File not found (already deleted): {x_file_path}")

# Delete y_labels.npy if it exists
if os.path.exists(y_file_path):
    os.remove(y_file_path)
    print(f"Deleted: {y_file_path}")
else:
    print(f"File not found (already deleted): {y_file_path}")

Deleted: /kaggle/working/X_data.npy
Deleted: /kaggle/working/y_labels.npy


In [60]:
def data_preparation():
    if os.path.exists(x_file_path) and os.path.exists(y_file_path):
        X_data = np.load(x_file_path)
        y_labels = np.load(y_file_path)
        return X_data,y_labels
    else:
        X_data = []
        y_labels  = []
        for dir_ in sorted(os.listdir(DATA_DIR)):
            dir_path = os.path.join(DATA_DIR, dir_)
            for img_path in os.listdir(dir_path)[:1500]:
                img = cv2.imread(os.path.join(dir_path, img_path))
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
                
                detection_result = detector.detect(mp_image)
                
                if detection_result.hand_landmarks:
                    hand_landmarks = detection_result.hand_landmarks[0]
                    
                    row = []
                    for lm in hand_landmarks:
                        row.extend([lm.x, lm.y, lm.z])
                    if len(row) == 63:
                        X_data.append(row)
                        y_labels.append(dir_)
            print(dir_," done")
    
        X_data = np.array(X_data)
        y_labels = np.array(y_labels)
    
    print("Data shape for X Data ANN:", X_data.shape)
    print("Data shape for Y Labels:", y_labels.shape)

    print("DATA SAVED IN THE WORKING DIRECTORY")
    np.save(x_file_path, X_data)
    np.save(y_file_path, y_labels)
    
    return X_data, y_labels

In [63]:
def train_model(X_data,y_labels):
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_labels)
    num_classes = len(label_encoder.classes_)
    
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=0.2, random_state=42)
    
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(63,)), # Accepts the 63 landmark inputs 
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(num_classes, activation='softmax') # Outputs probability distribution
    ])

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',      
        patience=25,              
        restore_best_weights=True
    )


    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    
    history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_data=(X_test, y_test),callbacks=[early_stopping])
    return history, model, label_encoder

In [44]:
def show_graph(history):
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, color='royalblue')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='darkorange')
    ax1.set_title('Model Accuracy Over Epochs', fontsize=14)
    ax1.set_xlabel('Epochs', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    ax2.plot(history.history['loss'], label='Train Loss', linewidth=2, color='royalblue')
    ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='darkorange')
    ax2.set_title('Model Loss Over Epochs', fontsize=14)
    ax2.set_xlabel('Epochs', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.legend(fontsize=11)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

In [69]:
print("-"*50)
print("PREPARING DATA: ")
X_data, y_labels = data_preparation()
print("-"*50)
print("TRAINING THE MODEL: ")
history,model,label_encoder = train_model(X_data,y_labels)
print("-"*50)
print("DRAWING GRAPHS: ")
show_graph(history)
print("-"*50)

--------------------------------------------------
PREPARING DATA: 
--------------------------------------------------
TRAINING THE MODEL: 
Epoch 1/180
828/828 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7653 - loss: 0.9190 - val_accuracy: 0.8011 - val_loss: 0.6313
Epoch 2/180
828/828 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9131 - loss: 0.3102 - val_accuracy: 0.7986 - val_loss: 0.5592
Epoch 3/180
828/828 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9213 - loss: 0.2622 - val_accuracy: 0.8054 - val_loss: 0.6508
Epoch 4/180
828/828 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9245 - loss: 0.2509 - val_accuracy: 0.8336 - val_loss: 0.4814
Epoch 5/180
828/828 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9282 - loss: 0.2340 - val_accuracy: 0.9322 - val_loss: 0.2065
Epoch 6/180
828/828 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9309 - loss: 0.2210 - val_accuracy: 0.8785 - val_loss: 0.3444
Epoch 7/180
828/828 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9398 - loss: 0.1997

TypeError: cannot unpack non-iterable History object

In [67]:
# 2. Extract the last index (the epoch where early stopping kicked in)
# Because restore_best_weights=True, index -1 points exactly to the best epoch's metrics!
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

# 3. Print them cleanly
print("-" * 40)
print(f"Final Training Accuracy:   {final_train_acc * 100:.2f}%")
print(f"Final Validation Accuracy: {final_val_acc * 100:.2f}%")
print(f"Final Training Loss:       {final_train_loss:.4f}")
print(f"Final Validation Loss:     {final_val_loss:.4f}")
print("-" * 40)

----------------------------------------
Final Training Accuracy:   95.39%
Final Validation Accuracy: 96.16%
Final Training Loss:       0.1468
Final Validation Loss:     0.1325
----------------------------------------


In [70]:

def predict_gesture(image_path, model, label_encoder):
    """
    Takes an image path, extracts landmarks using modern MediaPipe, 
    and predicts the sign language gesture using your trained ANN.
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not read image at {image_path}")
        return None
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

    detection_result = detector.detect(mp_image)
    
    # 4. If a hand is found, run it through the ANN
    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]
        
        row = []
        for lm in hand_landmarks:
            row.extend([lm.x, lm.y, lm.z])
            
        if len(row) == 63:
            # Convert to a NumPy array and reshape to (1, 63) because the ANN expects a batch dimension
            input_data = np.array([row])
            
            # 5. Make the prediction
            predictions = model.predict(input_data, verbose=0)
            
            # Find the index with the highest probability
            predicted_class_idx = np.argmax(predictions)
            confidence = predictions[0][predicted_class_idx]
            
            # Decode the integer index back into the original folder label (e.g., 'A', 'B')
            predicted_label = label_encoder.inverse_transform([predicted_class_idx])[0]
            
            return predicted_label, confidence
            
    print("No hand landmarks detected in this image.")
    return None, 0.0


In [72]:
for dir_ in sorted(os.listdir(DATA_DIR))[2:6]:
            dir_path = os.path.join(DATA_DIR, dir_)
            for img_path in os.listdir(dir_path)[1500:1502]:
                full_img_path = os.path.join(dir_path, img_path)
                label, conf = predict_gesture(full_img_path, model, label_encoder)
                print(f"Predicted Gesture: {label} ({conf * 100:.2f}% Confidence) for {dir_}")

No hand landmarks detected in this image.
Predicted Gesture: None (0.00% Confidence) for C
Predicted Gesture: C (99.92% Confidence) for C
Predicted Gesture: D (99.42% Confidence) for D
Predicted Gesture: O (98.65% Confidence) for D
Predicted Gesture: E (99.98% Confidence) for E
Predicted Gesture: E (78.05% Confidence) for E
Predicted Gesture: F (99.99% Confidence) for F
Predicted Gesture: F (99.79% Confidence) for F
